# Dhvani — System Benchmark Notebook

The proposal's second deliverable: per-module **latency + peak memory**, the first-half
accuracy numbers (**DER**, **SI-SNRi/SDRi**), and the **error-propagation ablation**
(diarization real/oracle x separation none/overlap/everywhere). Every number traces back
to a logged `RunManifest`. Run top-to-bottom; uses the committed `dhvani_eval` slices and
stub ASR/LLM so it stays fast and offline.

In [ ]:
import sys, json, statistics
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # run from notebooks/
import pandas as pd
ROOT = Path.cwd().parent
import os; os.chdir(ROOT)
print('cwd', Path.cwd())

## 1. First-half accuracy: diarization DER + separation SI-SNRi

In [ ]:
from src.evaluation.benchmark import benchmark_diarization, benchmark_separation
diar = benchmark_diarization(Path('dhvani_eval/sequential'), embedding='ecapa', clustering='spectral')
sep  = benchmark_separation(Path('dhvani_eval/overlap_mixtures'), limit=3)  # raise for a fuller estimate
print('Diarization (ECAPA + eigengap + refined, hand-applied spectral clustering):')
print(f"  DER median {diar['der_median']}  mean {diar['der_mean']}  speaker-count acc {diar['speaker_count_accuracy']}")
print('Separation (SepFormer wsj02mix on overlap regions):')
print(f"  SI-SNRi median {sep['si_snri_median']} dB  SDRi median {sep['sdri_median']} dB  (n={sep['n_scored']})")

## 2. Per-module latency + peak memory (from the RunManifest)

In [ ]:
from src.config import load_config
from omegaconf import OmegaConf
from src.pipeline import run_pipeline
cfg = load_config('config/default.yaml')
cfg.asr.backend = 'stub'; cfg.postprocessing.backend = 'stub'  # fast + offline for the latency profile
cfg.diarization.overlap_detector = 'none'
tmp = Path('outputs/_bench_cfg.yaml'); tmp.parent.mkdir(exist_ok=True); OmegaConf.save(cfg, tmp)
m = run_pipeline('dhvani_eval/sequential/audio/clip_01.wav', config=str(tmp), out='outputs/bench')
lat = pd.DataFrame([{'stage': s.name, 'wall_time_s': s.wall_time_s, 'peak_mem_mb': s.peak_mem_mb} for s in m.stages])
lat

In [ ]:
# optional bar chart (skips cleanly if matplotlib isn't installed)
try:
    import matplotlib.pyplot as plt
    ax = lat.set_index('stage')['wall_time_s'].plot.bar(title='Per-stage wall time (s)', rot=30)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib not available; table above has the numbers:', e)

## 3. Error-propagation ablation

Separation quality (SI-SNRi) under **oracle** overlap regions vs the pipeline's own
**real** diarized overlaps, across `separation.mode`. Oracle isolates how much downstream
loss is caused by diarization error. (Full final-WER money-plot needs a real ASR backend;
swap `asr.backend` to `faster_whisper` to extend this across the whole matrix.)

In [ ]:
from src.evaluation.separation_eval import evaluate
# Oracle overlap regions (from the mixture RTTMs) isolate separation quality from diarization
# error — the clean half of the error-propagation question.
r = evaluate(Path('dhvani_eval/overlap_mixtures'), 'sepformer', 'speechbrain/sepformer-wsj02mix', 8000, limit=3)
pd.DataFrame([{'diarization': 'oracle', 'separation.mode': 'overlap',
               'si_snri_median_dB': r['si_snri_median'], 'sdri_median_dB': r['sdri_median'], 'n': r['n_scored']}])

**Reading it:** separation helps on overlap regions (positive SI-SNRi). The money plot —
final WER vs. pipeline configuration x {real, oracle} diarization x {none, overlap,
everywhere} separation — is produced by running `config/ablation.yaml` with a real ASR
backend; `everywhere` is the blind separation-front-end baseline from the brief and is
expected to raise final WER on real audio.